# Verify the TransE `.sum()` fix (GPU, cell-by-cell)
Runs the fixed PyTorch TransE on the biokg rerun-0 and rerun-2 splits and checks AUROC recovers to ~0.95 (was ~0.85/0.4 with the `.mean()` bug). Select the GPU Jupyter kernel before running.

In [ ]:
# Cell 1 - make paths work whether cwd is repo root or eval_notebooks/
import os, sys
from pathlib import Path
root = Path.cwd()
if root.name == 'eval_notebooks':
    os.chdir(root.parent); root = root.parent
sys.path.insert(0, str(root))
print('cwd:', os.getcwd())

In [ ]:
# Cell 2 - confirm we are actually on the GPU
import torch
from src.embedding import _get_device
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
print('device:', _get_device())   # want: cuda

In [ ]:
# Cell 3 - the verification: fixed TransE on two known splits
import pickle, time
from src.embedding import TransE, compute_embedding_metrics
for rerun in (0, 2):
    p = pickle.load(open(f'results/cache/biokg_prep_rerun{rerun}.pkl', 'rb'))
    m = TransE(n_entities=p['n_ent'], n_relations=p['n_rels'],
               dim=128, margin=1.0, lr=0.01, seed=p['rerun_seed'])
    t = time.time()
    m.fit(p['train_triples'], n_epochs=100, batch_size=512, verbose=True)
    res = compute_embedding_metrics(m, p['test_pos'],
            p['neg_by_strategy']['type-constrained'], p['rel_idx'],
            rel_idx_inv=p['rel_idx_inv'])
    print(f'biokg rerun{rerun}  AUROC={res["auroc"]:.4f}  ({time.time()-t:.0f}s)')
print('Fix works if both are ~0.95 (NumPy reference was 0.956).')